In [1]:
!pip install -U "rfdetr[train,loggers]" supervision

In [2]:
from pathlib import Path
 
import cv2
import supervision as sv
import yaml
from rfdetr import RFDETRMedium  # also: RFDETRNano / RFDETRSmall / RFDETRBase / RFDETRLarge

In [3]:
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
ORIGINAL_DATA_YAML = "/kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11/data.yaml"
WRITABLE_DIR = "/kaggle/working"
OUTPUT_DIR = f"{WRITABLE_DIR}/rfdetr_run"
 
EPOCHS = 50
BATCH = 4            # per-step batch size (RF-DETR is memory hungry)
GRAD_ACCUM = 4       # effective batch = BATCH * GRAD_ACCUM = 16
LR = 1e-4
PATIENCE = 30
CONF = 0.25
 
 
# ---------------------------------------------------------------------------
# Dataset: Roboflow YOLO export (data.yaml + train/valid/test folders)
# RF-DETR reads this layout directly, so no rewritten yaml is needed.
# ---------------------------------------------------------------------------
def check_dataset():
    yaml_path = Path(ORIGINAL_DATA_YAML).resolve()
    root = yaml_path.parent
    for split in ("train", "valid", "test"):
        for sub in ("images", "labels"):
            d = root / split / sub
            if not d.exists():
                raise FileNotFoundError(f"Missing dir: {d}")
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    print(f"Dataset root: {root}")
    print(f"Classes ({cfg['nc']}): {cfg['names']}")
    return root, yaml_path, cfg
 
 

In [4]:
# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
def train(dataset_root):
    model = RFDETRMedium()  # COCO-pretrained; resolution is fixed by model size
    model.train(
        dataset_dir=str(dataset_root),
        epochs=EPOCHS,
        batch_size=BATCH,
        grad_accum_steps=GRAD_ACCUM,
        lr=LR,
        output_dir=OUTPUT_DIR,
        early_stopping=True,
        early_stopping_patience=PATIENCE,
    )
    # Best checkpoint (EMA usually the best; fall back to total if absent)
    ckpt = Path(OUTPUT_DIR) / "checkpoint_best_total.pth"
    if not ckpt.exists():
        ckpt = Path(OUTPUT_DIR) / "checkpoint_best_ema.pth"
    print(f"\nTraining complete. Best weights: {ckpt}")
    return str(ckpt)

In [ ]:
# ---------------------------------------------------------------------------
# Evaluation (supervision metrics)
# ---------------------------------------------------------------------------
def load_split(root, yaml_path, split):
    return sv.DetectionDataset.from_yolo(
        images_directory_path=str(root / split / "images"),
        annotations_directory_path=str(root / split / "labels"),
        data_yaml_path=str(yaml_path),
    )
 
 
def evaluate(model, ds, label, class_names):
    preds, targets = [], []
    for _path, image, gt in ds:
        det = model.predict(image, threshold=CONF)
        preds.append(det)
        targets.append(gt)
 
    map_metric = sv.metrics.MeanAveragePrecision()
    map_res = map_metric.update(preds, targets).compute()
 
    prec_res = sv.metrics.Precision().update(preds, targets).compute()
    rec_res = sv.metrics.Recall().update(preds, targets).compute()
 
    print(f"\n--- {label} ---")
    print(f"Precision (IoU .50): {prec_res.precision_at_50:.4f}")
    print(f"Recall    (IoU .50): {rec_res.recall_at_50:.4f}")
    print(f"mAP50:               {map_res.map50:.4f}")
    print(f"mAP50-95:            {map_res.map50_95:.4f}")
 
    # Per-class AP (columns of ap_per_class = IoU thresholds .50:.05:.95)
    print("\nPer-class")
    for row, cid in enumerate(map_res.matched_classes):
        ap = map_res.ap_per_class[row]
        print(f"{class_names[int(cid)]:<12} mAP50: {ap[0]:.3f}  mAP50-95: {ap.mean():.3f}")
 
 
# ---------------------------------------------------------------------------
# Annotated predictions on test images
# ---------------------------------------------------------------------------
def save_test_predictions(model, root, class_names):
    out_dir = Path(WRITABLE_DIR) / "predict"
    out_dir.mkdir(parents=True, exist_ok=True)
    box_ann, label_ann = sv.BoxAnnotator(), sv.LabelAnnotator()
 
    for img_path in sorted((root / "test" / "images").glob("*")):
        image = cv2.imread(str(img_path))
        if image is None:
            continue
        det = model.predict(image, threshold=CONF)
        labels = [f"{class_names[int(c)]} {s:.2f}" for c, s in zip(det.class_id, det.confidence)]
        image = box_ann.annotate(image.copy(), det)
        image = label_ann.annotate(image, det, labels=labels)
        cv2.imwrite(str(out_dir / img_path.name), image)
 
    print(f"\nAnnotated test predictions saved to: {out_dir}")
 
 
# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------
root, yaml_path, cfg = check_dataset()
class_names = cfg["names"]
 
best_ckpt = train(root)

model = RFDETRMedium(pretrain_weights=best_ckpt, num_classes=len(class_names))
 
evaluate(model, load_split(root, yaml_path, "valid"), "Validation (valid split)", class_names)
evaluate(model, load_split(root, yaml_path, "test"), "Test split", class_names)
save_test_predictions(model, root, class_names)
 

Dataset root: /kaggle/input/datasets/nomanhossain4/green-dataset/Vehicles.v5i.yolov11
Classes (7): ['Bus', 'Jeepney', 'Motorcycle', 'Tricycle', 'Van', 'cars', 'truck']
[2026-10-04 07:32:42] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


[2026-10-04 07:32:42] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-04 07:32:42] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-10-04 07:32:43] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.
